## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.feature_selection import mutual_info_classif
from scipy.cluster.hierarchy import linkage, leaves_list

from pymining import seqmining
from itertools import combinations
from prefixspan import PrefixSpan
from mlxtend.frequent_patterns import apriori, association_rules
import time

In [42]:
# pip install pymining prefixspan pandas mlxtend

## Task 2.1 Data Preprocessing

In [31]:
# Load data
data = pd.read_csv("data/cancer-data.csv")

# Drop column with all NaN values
data = data.drop(columns=["Unnamed: 32"], errors='ignore')
data.info()

# Separate features and label
X = data.drop(columns=["id", "diagnosis"])
y = data["diagnosis"].map({"M": 1, "B": 0})  # 1 = malignant, 0 = benign
feature_names = X.columns

# Normalize data for z-score operations
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [ ]:
# Define sequence generation functions
# --- 1. Z-score-based sequence ---
def make_sequence_zscore(x_row, top_k=8):
    """Rank features by absolute z-score (per patient)"""
    ranked = x_row.abs().sort_values(ascending=False)
    top_features = ranked.head(top_k).index.tolist()
    return [f"{feature}_high" if x_row[feature] > 0 else f"{feature}_low" for feature in top_features]

# --- 2. Mutual Information-based sequence ---
def make_sequence_mi(x_row, mi_scores, X_scaled, top_k=8):
    """Order features by global mutual information importance"""
    top_features = mi_scores.head(top_k).index.tolist()
    sequence = []
    for feature in top_features:
        val = x_row[feature]
        mean_val = X_scaled[feature].mean()
        label = "high" if val > mean_val else "low"
        sequence.append(f"{feature}_{label}")
    return sequence

# --- 3. Binned sequence ---
def make_sequence_binned(X, n_bins=3, strategy="quantile"):
    """Convert continuous values into low/medium/high bins"""
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy=strategy)
    X_binned = pd.DataFrame(discretizer.fit_transform(X), columns=X.columns)
    
    sequences = []
    labels = ['low', 'med', 'high']
    
    for _, row in X_binned.iterrows():
        seq = [f"{feature}_{labels[int(row[feature])]}" for feature in X.columns]
        sequences.append(seq)
    return sequences

# Apply different semantics
# Semantic 1: Z-score ranking per patient
seqs_z = X_scaled.apply(make_sequence_zscore, axis=1)

# Semantic 2: Mutual Information–based ranking
mi_scores = pd.Series(mutual_info_classif(X_scaled, y), index=X.columns).sort_values(ascending=False)
seqs_mi = X_scaled.apply(lambda row: make_sequence_mi(row, mi_scores, X_scaled, top_k=5), axis=1)

# Semantic 3: Binning (low/med/high)
strategies = ["uniform", "quantile", "kmeans"]
seqs_binned_all = {}

for strat in strategies:
    seqs_binned_all[strat] = make_sequence_binned(X_scaled, n_bins=3, strategy=strat)

# Combine into a DataFrame
sequences_df = pd.DataFrame({
    "id": data["id"],
    "diagnosis": data["diagnosis"],
    "seq_zscore": seqs_z,
    "seq_mi": seqs_mi,
    "seq_binned_uniform": seqs_binned_all["uniform"],
    "seq_binned_quantile": seqs_binned_all["quantile"],
    "seq_binned_kmeans": seqs_binned_all["kmeans"],
})

# Save results
sequences_df.to_csv("data/sequence_semantics.csv", index=False)
print("Generated sequence representations saved to data/sequence_semantics.csv")

Generated sequence representations saved to data/sequence_semantics.csv


## Task 2.2 Data Analysis

In [ ]:
sequences_df = pd.read_csv("data/sequence_semantics.csv")

semantics = [
    "seq_zscore",
    "seq_mi", 
    "seq_binned_uniform",
    "seq_binned_quantile",
    "seq_binned_kmeans"
]

def run_prefixspan(sequences, min_support=0.2):
    from prefixspan import PrefixSpan
    ps = PrefixSpan(sequences)
    patterns = ps.frequent(int(min_support * len(sequences)))
    return ["->".join(pat[1]) for pat in patterns if len(pat[1]) > 0]

def run_gsp(sequences, min_support=0.2, max_len=3):
    from collections import Counter
    freq_patterns = []
    counts = Counter()
    for seq in sequences:
        # Handle different length subsequences
        for l in range(1, min(len(seq), max_len) + 1):
            for combo in combinations(seq, l):
                counts[combo] += 1
    n = len(sequences)
    for seq, count in counts.items():
        if count / n >= min_support:
            freq_patterns.append("->".join(seq))
    return freq_patterns

def run_spade_like(sequences, min_support=0.2):
    try:
        # Create transaction matrix
        all_items = set()
        for seq in sequences:
            all_items.update(seq)
        
        transactions = []
        for seq in sequences:
            transaction = {item: 1 for item in seq}
            transactions.append(transaction)
        
        transactions_df = pd.DataFrame(transactions).fillna(0)
        
        if len(transactions_df.columns) > 0:
            freq = apriori(transactions_df, min_support=min_support, use_colnames=True)
            if len(freq) > 0:
                freq["pattern"] = freq["itemsets"].apply(lambda x: "->".join(sorted(list(x))))
                return freq["pattern"].tolist()
        return []
    except:
        return []

# --- Run comparisons ---
results = []

for sem in semantics:
    print(f"Processing semantic: {sem}")
    sequences = sequences_df[sem].apply(eval).tolist()
    
    for algo_name, algo_func in [
        ("GSP", run_gsp),
        ("PrefixSpan", run_prefixspan), 
        ("SPADE", run_spade_like),
    ]:
        print(f"  Running {algo_name}...")
        start = time.time()
        try:
            patterns = algo_func(sequences, min_support=0.15)  # Lower support threshold
            elapsed = time.time() - start
            
            results.append({
                "semantic": sem,
                "algorithm": algo_name,
                "num_patterns": len(patterns),
                "avg_len": sum(len(p.split("->")) for p in patterns)/len(patterns) if patterns else 0,
                "runtime_sec": round(elapsed, 3),
                "example_patterns": patterns[:3] if patterns else []
            })
        except Exception as e:
            print(f"    Error in {algo_name}: {e}")
            results.append({
                "semantic": sem,
                "algorithm": algo_name,
                "num_patterns": 0,
                "avg_len": 0,
                "runtime_sec": 0,
                "example_patterns": []
            })

results_df = pd.DataFrame(results)
print("\n=== ALGORITHM PERFORMANCE COMPARISON ===")
print(results_df[['semantic', 'algorithm', 'num_patterns', 'avg_len', 'runtime_sec']])


In [ ]:
# --- Comprehensive Analysis ---
print("="*80)
print("                    COMPREHENSIVE ANALYSIS")
print("="*80)

# 1. Binning Strategy Comparison
print("\n1. BINNING STRATEGY COMPARISON:")
binning_results = []
for strategy in ["uniform", "quantile", "kmeans"]:
    col_name = f"seq_binned_{strategy}"
    sequences = sequences_df[col_name].apply(eval).tolist()
    patterns = run_gsp(sequences, min_support=0.15)
    binning_results.append({
        'strategy': strategy,
        'num_patterns': len(patterns),
        'avg_length': np.mean([len(p.split("->")) for p in patterns]) if patterns else 0
    })

binning_df = pd.DataFrame(binning_results)
print(binning_df)

# 2. Support Threshold Sensitivity
print("\n2. SUPPORT THRESHOLD SENSITIVITY:")
support_results = []
for support in [0.1, 0.15, 0.2, 0.25, 0.3]:
    sequences = sequences_df['seq_zscore'].apply(eval).tolist()
    patterns = run_gsp(sequences, min_support=support)
    support_results.append({
        'min_support': support,
        'num_patterns': len(patterns)
    })

support_df = pd.DataFrame(support_results)
print(support_df)

# 3. Diagnosis-Specific Patterns (top 5 each)
print("\n3. DIAGNOSIS-SPECIFIC PATTERNS:")
sequences = sequences_df['seq_zscore'].apply(eval).tolist()
diagnoses = sequences_df['diagnosis'].tolist()

malignant_seqs = [seq for seq, diag in zip(sequences, diagnoses) if diag == 'M']
benign_seqs = [seq for seq, diag in zip(sequences, diagnoses) if diag == 'B']

mal_patterns = run_gsp(malignant_seqs, min_support=0.15)[:5]
ben_patterns = run_gsp(benign_seqs, min_support=0.15)[:5]

print(f"Malignant patterns: {mal_patterns}")
print(f"Benign patterns: {ben_patterns}")

# 4. Feature Importance (top 10)
print("\n4. TOP 10 FEATURES BY MUTUAL INFORMATION:")
print(mi_scores.head(10))

print("\n" + "="*80)

In [ ]:
# --- Final Results Summary ---
print("="*80)
print("                    FINAL RESULTS SUMMARY")
print("="*80)

# Best performing configurations
print("\n1. BEST CONFIGURATIONS:")
best_patterns = results_df.loc[results_df['num_patterns'].idxmax()]
fastest = results_df.loc[results_df['runtime_sec'].idxmin()]

print(f"Most patterns: {best_patterns['semantic']} + {best_patterns['algorithm']} ({best_patterns['num_patterns']} patterns)")
print(f"Fastest: {fastest['semantic']} + {fastest['algorithm']} ({fastest['runtime_sec']}s)")

# Algorithm performance summary
print("\n2. ALGORITHM PERFORMANCE SUMMARY:")
perf_summary = results_df.groupby('algorithm').agg({
    'num_patterns': 'mean',
    'runtime_sec': 'mean'
}).round(3)
print(perf_summary)

# Semantic method effectiveness
print("\n3. SEMANTIC METHOD EFFECTIVENESS:")
semantic_summary = results_df.groupby('semantic').agg({
    'num_patterns': 'mean',
    'runtime_sec': 'mean'
}).round(3)
print(semantic_summary)
print("\n" + "="*80)